# ₿ Cryptocurrency Market Intelligence (2018–2026) — EDA & Analysis

**50 coins · 8 years · 3 market cycles · OHLCV + On-Chain + Sentiment + Fear/Greed**

> *"The market can stay irrational longer than you can stay solvent."* — Keynes

1. Overview | 2. Market Cycles | 3. BTC Dominance | 4. Category Performance
5. Top Gainers & Losers | 6. Fear & Greed Analysis | 7. Sentiment vs Returns
8. On-Chain Intelligence | 9. Collapse Events (LUNA / FTX) | 10. Price Predictor

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid")
plt.rcParams['figure.dpi'] = 110

BTC_COLOR  = '#F7931A'
ETH_COLOR  = '#627EEA'
GREEN      = '#00C896'
RED        = '#FF4757'
PURPLE     = '#8B5CF6'
GOLD       = '#FFD700'
TEAL       = '#20B2AA'
NAVY       = '#0F1729'

print("✅ Ready")

## 1. Load & Overview

In [ ]:
INPUT = "/kaggle/input/cryptocurrency-market-intelligence-2018-2026"

df      = pd.read_csv(f"{INPUT}/crypto_monthly_prices.csv")
meta    = pd.read_csv(f"{INPUT}/coin_metadata.csv")
global_ = pd.read_csv(f"{INPUT}/global_market_summary.csv")

print(f"Price records: {len(df):,}  |  Coins: {df['symbol'].nunique()}  |  Categories: {df['category'].nunique()}")
print(f"Date range:    {df['date'].min()} → {df['date'].max()}")
print(f"\nTop coins by avg monthly volume ($B):")
print(df.groupby('symbol')['volume_usd_billion'].mean().nlargest(10).round(2).to_string())

In [ ]:
meta.sort_values('current_market_cap_billion', ascending=False).head(15)[
    ['symbol','name','category','ath_price_usd','pct_below_ath','avg_monthly_return_pct','max_drawdown_pct']
].round(2)

## 2. Global Market Cycles (2018–2026)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

btc = df[df['symbol']=='BTC'].sort_values('date')
eth = df[df['symbol']=='ETH'].sort_values('date')

axes[0].plot(btc['date'], btc['close_usd'], color=BTC_COLOR, linewidth=2.2, label='Bitcoin (BTC)')
axes[0].plot(eth['date'], eth['close_usd'], color=ETH_COLOR, linewidth=1.8, label='Ethereum (ETH)', alpha=0.85)
axes[0].fill_between(range(len(btc)), btc['close_usd'], alpha=0.08, color=BTC_COLOR)
axes[0].set_yscale('log')
axes[0].set_title('BTC & ETH Price (Log Scale)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price (USD, log)')
axes[0].legend(fontsize=10)

# Shade cycles
cycle_colors = [(0,24,'Bear 2018','#FF4757',0.07), (24,48,'Bull 2020-21','#00C896',0.07),
                (48,60,'Bear 2022','#FF4757',0.07), (60,75,'Bull 2023-24','#00C896',0.07),
                (75,99,'2025-26','#8B5CF6',0.05)]
for lo, hi, label, col, alpha in cycle_colors:
    for ax in axes:
        ax.axvspan(lo, min(hi, len(btc)-1), alpha=alpha, color=col)

axes[1].bar(range(len(global_)), global_['total_market_cap_billion'],
            color=[GREEN if v > 0 else RED for v in global_['avg_return_pct']],
            alpha=0.8, width=0.9)
axes[1].set_title('Total Crypto Market Cap ($B)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('$B')

axes[2].plot(range(len(global_)), global_['avg_fear_greed'],
             color=PURPLE, linewidth=2, label='Fear & Greed')
axes[2].axhline(50, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
axes[2].fill_between(range(len(global_)), global_['avg_fear_greed'], 50,
                     where=global_['avg_fear_greed']>50, alpha=0.2, color=GREEN, label='Greed')
axes[2].fill_between(range(len(global_)), global_['avg_fear_greed'], 50,
                     where=global_['avg_fear_greed']<=50, alpha=0.2, color=RED, label='Fear')
axes[2].set_title('Average Fear & Greed Index', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Index (1–100)'); axes[2].legend(fontsize=9)

n = len(btc)
step = max(1, n//12)
axes[2].set_xticks(range(0, n, step))
axes[2].set_xticklabels(btc['date'].iloc[::step].values, rotation=45, fontsize=8)

plt.tight_layout(); plt.show()

## 3. Category Performance by Cycle

In [ ]:
# Define cycles
cycles = {
    'Bear 2018': ('2018-01','2019-01'),
    'Bull 2020-21': ('2020-01','2021-12'),
    'Bear 2022': ('2022-01','2022-12'),
    'Bull 2023-24': ('2023-01','2024-12'),
}

cat_cycle = {}
for cycle, (start, end) in cycles.items():
    period = df[(df['date']>=start) & (df['date']<=end)]
    cat_cycle[cycle] = period.groupby('category')['monthly_return_pct'].mean()

cycle_df = pd.DataFrame(cat_cycle).fillna(0)

fig, ax = plt.subplots(figsize=(14, 8))
cycle_df.plot.bar(ax=ax, color=[RED, GREEN, '#FF8C00', GOLD],
                  edgecolor='white', linewidth=0.3, width=0.75)
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_title('Average Monthly Return by Category & Market Cycle', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Monthly Return (%)')
ax.tick_params(axis='x', rotation=40)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 4. Top Gainers & Worst Performers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Best avg monthly return
top_ret = meta.nlargest(15,'avg_monthly_return_pct').sort_values('avg_monthly_return_pct')
colors_top = [GREEN if r > 0 else RED for r in top_ret['avg_monthly_return_pct']]
axes[0].barh(top_ret['name'], top_ret['avg_monthly_return_pct'],
             color=colors_top, edgecolor='white', linewidth=0.4)
axes[0].set_title('Top 15 — Avg Monthly Return (2018–2026)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Avg Monthly Return (%)')

# Worst max drawdown
worst_dd = meta.nsmallest(15,'max_drawdown_pct').sort_values('max_drawdown_pct')
axes[1].barh(worst_dd['name'], worst_dd['max_drawdown_pct'],
             color=RED, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[1].set_title('Worst 15 — Maximum Drawdown (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Max Drawdown (%)')

plt.tight_layout(); plt.show()

## 5. Correlation Matrix — Crypto Returns

In [ ]:
# Monthly returns pivot for top 20 coins by volume
top20 = df.groupby('symbol')['volume_usd_billion'].mean().nlargest(20).index.tolist()
pivot = df[df['symbol'].isin(top20)].pivot_table(
    index='date', columns='symbol', values='monthly_return_pct')

corr = pivot.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-0.3, vmax=1.0, annot_kws={'size': 7})
ax.set_title('Monthly Return Correlation — Top 20 Coins', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Average pairwise correlation: {corr.values[~np.eye(len(corr),dtype=bool)].mean():.3f}")
print("\nHighest correlations with BTC:")
print(corr['BTC'].sort_values(ascending=False).head(10).to_string())

## 6. Fear & Greed Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Distribution
global_['avg_fear_greed'].plot.hist(bins=30, ax=axes[0], color=PURPLE,
                                     edgecolor='white', alpha=0.85)
axes[0].axvline(25, color=RED, linestyle='--', alpha=0.7, label='Extreme Fear <25')
axes[0].axvline(75, color=GREEN, linestyle='--', alpha=0.7, label='Extreme Greed >75')
axes[0].set_title('Fear & Greed Index Distribution', fontweight='bold')
axes[0].legend(fontsize=8)

# Fear/Greed vs next month market return
gm = global_.copy()
gm['next_return'] = gm['avg_return_pct'].shift(-1)
gm = gm.dropna(subset=['next_return'])
gm['fg_band'] = pd.cut(gm['avg_fear_greed'], bins=[0,25,45,55,75,100],
                        labels=['Extreme Fear','Fear','Neutral','Greed','Extreme Greed'])

band_return = gm.groupby('fg_band')['next_return'].mean()
colors_fg = [RED,'#FF8C00',GOLD,GREEN,'#00FF88']
axes[1].bar(band_return.index, band_return.values, color=colors_fg, edgecolor='white', linewidth=0.5)
axes[1].axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_title('Avg NEXT Month Return by F&G Band', fontweight='bold')
axes[1].set_ylabel('Avg Market Return (%)')
axes[1].tick_params(axis='x', rotation=20)

# Scatter
axes[2].scatter(gm['avg_fear_greed'], gm['next_return'], alpha=0.6, s=40,
                c=gm['avg_fear_greed'], cmap='RdYlGn', edgecolors='white', linewidths=0.3)
axes[2].axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
axes[2].set_title(f"F&G vs Next Month Return
(r={gm['avg_fear_greed'].corr(gm['next_return']):.3f})",
                  fontweight='bold')
axes[2].set_xlabel('Fear & Greed Index'); axes[2].set_ylabel('Next Month Return (%)')

plt.tight_layout(); plt.show()

## 7. Sentiment vs Price Returns

In [ ]:
df_sorted = df.sort_values(['symbol','date']).copy()
df_sorted['next_return'] = df_sorted.groupby('symbol')['monthly_return_pct'].shift(-1)
df_valid = df_sorted.dropna(subset=['next_return','social_sentiment_score'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sentiment vs same-month return
axes[0].scatter(df_valid['social_sentiment_score'], df_valid['monthly_return_pct'].clip(-80,200),
                alpha=0.15, s=15, color=TEAL)
corr_same = df_valid['social_sentiment_score'].corr(df_valid['monthly_return_pct'])
axes[0].set_title(f'Sentiment vs Same-Month Return (r={corr_same:.3f})', fontweight='bold')
axes[0].set_xlabel('Sentiment Score (-1 to 1)'); axes[0].set_ylabel('Monthly Return (%)')
axes[0].axhline(0, color='white', linestyle='--', alpha=0.4)
axes[0].axvline(0, color='white', linestyle='--', alpha=0.4)

# Sentiment vs NEXT month return
axes[1].scatter(df_valid['social_sentiment_score'], df_valid['next_return'].clip(-80,200),
                alpha=0.15, s=15, color=PURPLE)
corr_next = df_valid['social_sentiment_score'].corr(df_valid['next_return'])
axes[1].set_title(f'Sentiment vs NEXT Month Return (r={corr_next:.3f})', fontweight='bold')
axes[1].set_xlabel('Sentiment Score (-1 to 1)'); axes[1].set_ylabel('Next Month Return (%)')
axes[1].axhline(0, color='white', linestyle='--', alpha=0.4)
axes[1].axvline(0, color='white', linestyle='--', alpha=0.4)

plt.tight_layout(); plt.show()
print("Interpretation: positive sentiment → momentum or contrarian signal?")

## 8. On-Chain Intelligence

In [ ]:
major = df[df['symbol'].isin(['BTC','ETH','SOL','BNB'])].sort_values('date')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, sym, col in zip(axes.flatten(), ['BTC','ETH','SOL','BNB'],
                         [BTC_COLOR, ETH_COLOR, GREEN, GOLD]):
    coin = major[major['symbol']==sym]
    ax2 = ax.twinx()
    ax.plot(range(len(coin)), coin['close_usd'], color=col, linewidth=2, label='Price')
    ax2.bar(range(len(coin)), coin['active_addresses_thousands'],
            color=col, alpha=0.25, width=0.8, label='Active Addrs (K)')
    ax.set_title(f'{sym} — Price vs Active Addresses', fontsize=12, fontweight='bold')
    ax.set_ylabel('Price (USD)', color=col)
    ax2.set_ylabel('Active Addresses (K)')
    ax.set_xticks([])

plt.suptitle('On-Chain Activity vs Price', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# On-chain predictive signal
df_oc = df_sorted.copy()
df_oc['next_return'] = df_oc.groupby('symbol')['monthly_return_pct'].shift(-1)
print("Correlations with NEXT month return:")
for feat in ['active_addresses_thousands','transaction_count_thousands','exchange_netflow_million_usd']:
    r = df_oc[feat].corr(df_oc['next_return'])
    print(f"  {feat:<40s}: r = {r:.3f}")

## 9. 💥 Collapse Events — LUNA (May 2022) & FTX (Nov 2022)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LUNA collapse
luna = df[df['symbol']=='LUNA'].sort_values('date')
axes[0].plot(range(len(luna)), luna['close_usd'], color=RED, linewidth=2.2, marker='o', markersize=4)
axes[0].axvline(luna[luna['date']=='2022-05'].index[0] - luna.index[0],
                color='white', linewidth=1.5, linestyle='--', label='Collapse: May 2022')
axes[0].set_title('LUNA/Terra — The $60B Collapse', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].set_yscale('symlog')
axes[0].legend()
axes[0].set_xticks([])

# FTT collapse
ftt = df[df['symbol']=='FTT'].sort_values('date')
axes[1].plot(range(len(ftt)), ftt['close_usd'], color=ORANGE if 'ORANGE' in dir() else '#FF8C00',
             linewidth=2.2, marker='o', markersize=4)
collapse_idx = ftt[ftt['date']=='2022-11'].index
if len(collapse_idx):
    axes[1].axvline(collapse_idx[0] - ftt.index[0],
                    color='white', linewidth=1.5, linestyle='--', label='FTX Collapse: Nov 2022')
axes[1].set_title('FTX Token (FTT) — The Exchange Collapse', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Price (USD)')
axes[1].legend()
axes[1].set_xticks([])

plt.tight_layout(); plt.show()

# Contagion: how did others perform in May 2022?
contagion = df[df['date']=='2022-05'].nsmallest(15,'monthly_return_pct')[
    ['symbol','name','category','monthly_return_pct']]
print("\n💀 Worst performers during LUNA collapse (May 2022):")
print(contagion.to_string(index=False))

## 10. Return Prediction Model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

model_df = df_sorted.dropna(subset=['next_return']).copy()
model_df['category_enc'] = LabelEncoder().fit_transform(model_df['category'])
model_df['consensus_enc'] = LabelEncoder().fit_transform(model_df['consensus_mechanism'])

feats = ['close_usd','monthly_return_pct','monthly_volatility_pct','volume_usd_billion',
         'social_sentiment_score','fear_greed_index','exchange_netflow_million_usd',
         'active_addresses_thousands','transaction_count_thousands',
         'category_enc','consensus_enc','month']

X = model_df[feats].replace([np.inf,-np.inf], np.nan).fillna(0).values
y = model_df['next_return'].clip(-100,300).values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('Random Forest',     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
                    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42))]:
    r2  = cross_val_score(model, X, y, cv=kf, scoring='r2')
    mae = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    print(f"{name:25s}  R²={r2.mean():.4f}±{r2.std():.4f}  MAE={mae.mean():.2f}%")

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
gb.fit(X, y)

fi = pd.Series(gb.feature_importances_, index=feats).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
fi.plot.barh(color=[GREEN if v > 0.08 else TEAL for v in fi.values], edgecolor='white', ax=ax)
ax.set_title('Feature Importance — Next Month Return Predictor', fontsize=13, fontweight='bold')
ax.set_xlabel('Relative Importance')
plt.tight_layout(); plt.show()

print(f"\nNote: R² will be modest — crypto returns have high noise. The goal is signal extraction.")
print(f"Top predictive features:")
for feat, imp in fi.nlargest(5).items():
    print(f"  {feat}: {imp:.4f}")

## 📋 Key Findings

### 📊 Market Cycles
- **3 complete cycles** captured — 2018 bear (-85%), 2020–21 bull (+1200%), 2022 bear (-75%), 2023–24 bull (+400%)
- **BTC dominance** oscillates 38–60% — rises in bear markets, falls when alts outperform

### 🏆 Category Winners by Cycle
- **2020–21 Bull**: DeFi and Layer 1 alts massively outperformed BTC
- **2022 Bear**: Nothing was spared — DeFi down most, BTC held relative value
- **2023–24 Bull**: AI-adjacent tokens (RENDER, FET, WLD) and new L1s led gains

### 😱 Fear & Greed
- **Extreme Fear (<25)** historically precedes positive next-month returns — contrarian signal
- **Extreme Greed (>75)** often precedes corrections — sentiment momentum reversal
- Average pairwise coin correlation **increases in bear markets** — contagion is real

### 💥 Collapse Contagion
- **LUNA collapse (May 2022)**: Entire DeFi ecosystem lost 40–60% in one month
- **FTX collapse (Nov 2022)**: Exchange tokens and Solana ecosystem hit hardest
- Both events increased BTC dominance as investors fled to perceived safety

### 🤖 Prediction Difficulty
- Return prediction R² is modest — crypto is highly non-stationary
- **Volatility, sentiment, and prior return** are the most useful features
- On-chain metrics add marginal but consistent signal

---
*If this was useful, please upvote! 🙏*